In [1]:
# setup pyspark environment
import os
import sys

if sys.platform == "darwin":
    os.environ["SPARK_HOME"] = "/Users/hubert/Documents/dev/python/data-analysis-pyspark/spark-4.0.0-bin-hadoop3"
    os.environ["PATH"] += os.pathsep + "/Users/hubert/Documents/dev/python/data-analysis-pyspark/spark-4.0.0-bin-hadoop3/bin"

In [2]:
%pip install pandas
%pip install pyarrow


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("MyApp") \
    .config("spark.driver.extraJavaOptions", "-Dio.netty.tryReflectionSetAccessible=true") \
    .config("spark.executor.extraJavaOptions", "-Dio.netty.tryReflectionSetAccessible=true") \
    .config("spark.sql.execution.arrow.maxRecordsPerBatch", 10000) \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/09 18:31:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
import pyspark.sql.functions as F 
import pandas as pd

df = spark.createDataFrame(pd.DataFrame({'hi': ['hello', 'hi', 'hey']}))

df.select(F.upper(F.col('hi'))).show()

+---------+
|upper(hi)|
+---------+
|    HELLO|
|       HI|
|      HEY|
+---------+



In [5]:
import pyspark.sql.types as T
df2 = spark.createDataFrame(pd.DataFrame({'temps': [32.0, 98.0, 68.0]}), schema=T.StructType([T.StructField("temps", T.DoubleType(), True)]))

@F.pandas_udf(T.DoubleType())
def f_to_c(temps: pd.Series) -> pd.Series:
    return (temps - 32) * 5.0/9.0 

df2.select(f_to_c(F.col('temps')).alias('temps_celsius')).show()

df2.show()

df2.withColumn('temps_celsius', f_to_c(F.col('temps'))).show()

df2.show()

+------------------+
|     temps_celsius|
+------------------+
|               0.0|
|36.666666666666664|
|              20.0|
+------------------+

+-----+
|temps|
+-----+
| 32.0|
| 98.0|
| 68.0|
+-----+

+-----+------------------+
|temps|     temps_celsius|
+-----+------------------+
| 32.0|               0.0|
| 98.0|36.666666666666664|
| 68.0|              20.0|
+-----+------------------+

+-----+
|temps|
+-----+
| 32.0|
| 98.0|
| 68.0|
+-----+



In [6]:
from time import sleep 
from typing import Iterator 

@F.pandas_udf(T.DoubleType()) 
def f_to_c_batch(temps: Iterator[pd.Series]) -> Iterator[pd.Series]:
    sleep(5)
    for batch in temps:
        yield (batch - 32) * 5.0/9.0


df2.withColumn('temps_celsius', f_to_c_batch(F.col('temps'))).show()

+-----+------------------+
|temps|     temps_celsius|
+-----+------------------+
| 32.0|               0.0|
| 98.0|36.666666666666664|
| 68.0|              20.0|
+-----+------------------+



In [7]:
df3 = spark.createDataFrame(pd.DataFrame({"year": [2020, 2022, 2024], "month": [1, 6, 12], "day": [15, 20, 25]})
                            , schema=T.StructType([T.StructField("year", T.IntegerType(), True),
                                                  T.StructField("month", T.IntegerType(), True),
                                                  T.StructField("day", T.IntegerType(), True)]))
df3.show()

+----+-----+---+
|year|month|day|
+----+-----+---+
|2020|    1| 15|
|2022|    6| 20|
|2024|   12| 25|
+----+-----+---+



In [8]:
from typing import Tuple 

@F.pandas_udf(T.DateType())
def create_date(year_mo_da: Iterator[Tuple[pd.Series, pd.Series, pd.Series]]) -> Iterator[pd.Series]:
    for year, month, day in year_mo_da:
        yield pd.to_datetime({'year': year, 'month': month, 'day': day})

df3.withColumn('date', create_date(F.col('year'), F.col('month'), F.col('day'))).show()

+----+-----+---+----------+
|year|month|day|      date|
+----+-----+---+----------+
|2020|    1| 15|2020-01-15|
|2022|    6| 20|2022-06-20|
|2024|   12| 25|2024-12-25|
+----+-----+---+----------+



In [9]:
#series udf without decorators

exo_9_1 = pd.Series(["red", "blue", "blue", "yellow"])

def color_to_num(color_series: pd.Series) -> pd.Series:
    return color_series.map({"red": 1, "blue": 2, "yellow": 3})

print(color_to_num(exo_9_1))

color_to_num_udf = F.pandas_udf(color_to_num, returnType=T.IntegerType())

def color_to_num2(color_series: pd.Series) -> pd.Series:
    return color_series.apply(
        lambda c: {"red": 1, "blue": 2, "yellow": 3}.get(c, 0)
    )
color_to_num2_udf = F.pandas_udf(color_to_num2, returnType=T.IntegerType())
print(color_to_num2(exo_9_1))

0    1
1    2
2    2
3    3
dtype: int64
0    1
1    2
2    2
3    3
dtype: int64


Grouped Aggregator UDFs

In [10]:
%pip install scikit-learn
from sklearn.linear_model import LinearRegression


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [11]:
def rate_of_change_temp(day: pd.Series, temp: pd.Series) -> float:
    model = LinearRegression()
    X = day.astype(int).values.reshape(-1, 1) # convert to integer np array and reshape to a single column for sklearn
    y = temp.astype(float).values
    model.fit(X, y)
    return model.coef_[0]

rate_of_change_temp_udf = F.pandas_udf(rate_of_change_temp, returnType=T.DoubleType())

In [12]:
test_temps = pd.DataFrame({
    "day": [1, 2, 3, 4, 5],
    "temp": [30.0, 32.0, 34.0, 36.0, 38.0]
})

In [13]:
print(rate_of_change_temp(test_temps['day'], test_temps['temp']))

2.0


In [14]:
import utils.gsodUtils as gsodUtils
import os

gsod69015093121DataPath = "data/69015093121.csv"
gsod70000126492DataPath = "data/70000126492.csv"

if (not os.path.exists(gsod69015093121DataPath) or
    not os.path.exists(gsod70000126492DataPath)):
    gsodUtils.download_gsod_data("69015093121", 2024, gsod69015093121DataPath)
    gsodUtils.download_gsod_data("70000126492", 2024, gsod70000126492DataPath)



In [15]:
gsod6 = spark.read.csv(gsod69015093121DataPath, header=True, inferSchema=True)
gsod7 = spark.read.csv(gsod70000126492DataPath, header=True, inferSchema=True)

gsod = gsod6.union(gsod7)
gsod.printSchema()
gsod.show(5)

root
 |-- STATION: long (nullable = true)
 |-- DATE: date (nullable = true)
 |-- LATITUDE: double (nullable = true)
 |-- LONGITUDE: double (nullable = true)
 |-- ELEVATION: double (nullable = true)
 |-- NAME: string (nullable = true)
 |-- TEMP: double (nullable = true)
 |-- TEMP_ATTRIBUTES: double (nullable = true)
 |-- DEWP: double (nullable = true)
 |-- DEWP_ATTRIBUTES: double (nullable = true)
 |-- SLP: double (nullable = true)
 |-- SLP_ATTRIBUTES: double (nullable = true)
 |-- STP: double (nullable = true)
 |-- STP_ATTRIBUTES: double (nullable = true)
 |-- VISIB: double (nullable = true)
 |-- VISIB_ATTRIBUTES: double (nullable = true)
 |-- WDSP: double (nullable = true)
 |-- WDSP_ATTRIBUTES: double (nullable = true)
 |-- MXSPD: double (nullable = true)
 |-- GUST: double (nullable = true)
 |-- MAX: double (nullable = true)
 |-- MAX_ATTRIBUTES: string (nullable = true)
 |-- MIN: double (nullable = true)
 |-- MIN_ATTRIBUTES: string (nullable = true)
 |-- PRCP: double (nullable = true)

25/11/09 18:32:12 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [25]:
import pyspark.sql.functions as F

gsod = gsod.select(F.col("DATE"), F.col("STATION").alias("STN"), F.col("TEMP")) \
    .withColumn("year", F.year(F.col("DATE"))) \
    .withColumn("month", F.month(F.col("DATE"))) \
    .withColumn("day", F.day(F.col("DATE")))

gsod.show(2)


+----------+-----------+----+----+-----+---+
|      DATE|        STN|TEMP|year|month|day|
+----------+-----------+----+----+-----+---+
|2024-01-01|69015093121|46.5|2024|    1|  1|
|2024-01-02|69015093121|49.3|2024|    1|  2|
+----------+-----------+----+----+-----+---+
only showing top 2 rows


In [27]:
result = gsod.groupBy(F.col("STN"), F.col("year"), F.col("month")) \
    .agg(rate_of_change_temp_udf(gsod["day"], gsod["TEMP"])) \
    .alias("rate_of_change_temp")

result.show()

+-----------+----+-----+------------------------------+
|        STN|year|month|rate_of_change_temp(day, TEMP)|
+-----------+----+-----+------------------------------+
|69015093121|2024|    1|            0.7217338709677417|
|69015093121|2024|    2|            0.3866995073891626|
|69015093121|2024|    3|           0.08088709677419348|
|69015093121|2024|    4|            0.6100333704115681|
|69015093121|2024|    5|             0.452540322580645|
|69015093121|2024|    6|           0.19842046718576178|
|69015093121|2024|    7|          -0.22604838709677424|
|69015093121|2024|    8|          -0.35701612903225793|
|69015093121|2024|    9|          -0.44769744160177954|
|69015093121|2024|   10|           -0.8571488135163068|
|69015093121|2024|   11|           -0.2146384872080088|
|69015093121|2024|   12|           0.07157258064516125|
|70000126492|2024|    1|           -1.5063709677419348|
|70000126492|2024|    2|           0.47259870897105427|
|70000126492|2024|    3|            0.7335080645

In [28]:
def scale_temp(temp_by_day: pd.DataFrame) -> pd.DataFrame:
    """ Returns a simple normalization of the temperature for a site
    If the temperature is constant for the whole window, default to 0.5
    """
    temp = temp_by_day['TEMP']
    answer = temp_by_day[["STN", "year", "month", "day", "TEMP"]]
    if(temp.min() == temp.max()):
        return answer.assign(temp_norm = 0.5)
    return answer.assign(
        temp_norm = (temp - temp.min()) / (temp.max() - temp.min())
    )

In [36]:
gsod.show(2)
gsod_map = gsod.groupBy(F.col("STN"), F.col("year"), F.col("month")).applyInPandas(
    scale_temp,
    schema=T.StructType([
        T.StructField("STN", T.IntegerType(), True),
        T.StructField("year", T.IntegerType(), True),
        T.StructField("month", T.IntegerType(), True),
        T.StructField("day", T.IntegerType(), True),
        T.StructField("TEMP", T.DoubleType(), True),
        T.StructField("temp_norm", T.DoubleType(), True),
    ])
)
gsod_map.show(5)

+----------+-----------+----+----+-----+---+
|      DATE|        STN|TEMP|year|month|day|
+----------+-----------+----+----+-----+---+
|2024-01-01|69015093121|46.5|2024|    1|  1|
|2024-01-02|69015093121|49.3|2024|    1|  2|
+----------+-----------+----+----+-----+---+
only showing top 2 rows
+---------+----+-----+---+----+-------------------+
|      STN|year|month|day|TEMP|          temp_norm|
+---------+----+-----+---+----+-------------------+
|295616385|2024|    1|  1|46.5|0.18518518518518517|
|295616385|2024|    1|  2|49.3|0.27946127946127935|
|295616385|2024|    1|  3|47.2|0.20875420875420883|
|295616385|2024|    1|  4|48.8| 0.2626262626262625|
|295616385|2024|    1|  5|47.7|0.22558922558922567|
+---------+----+-----+---+----+-------------------+
only showing top 5 rows
